# Stage A2 — DeiT-Small

Một Transformer, dưới đúng pipeline đã dùng cho hai CNN. Cùng manifest, cùng
fold, cùng preprocessing, cùng loss — chỉ kiến trúc và lịch optimizer thay đổi.

---

## Mốc so sánh (mức filename-group, known benchmark)

| | AUC | Độ nhạy | Độ đặc hiệu | FP |
|---|---:|---:|---:|---:|
| ResNet18 B1 | 0,9779 | 0,9951 | 0,7689 | 52 |
| DenseNet121 v5 | 0,9801 | 0,9951 | 0,8044 | 44 |
| Ensemble R+D | 0,9792 | 0,9951 | **0,8222** | 40 |

Mục tiêu: **≥0,85 độ đặc hiệu** ở độ nhạy ≥0,97, tức FP ≤33 trên 225 group
NORMAL — cần giảm thêm 7 ca so với ensemble hiện tại.

## Quy tắc chọn checkpoint v6

```
1. độ đặc hiệu @ độ nhạy ≥97%     (endpoint chính)
2. hòa nếu |Δ| < 0,005
3. HSAS@97 cao hơn
4. hòa nếu |Δ| < 0,002
5. NLL không trọng số thấp hơn
6. epoch sớm hơn
```

**HSAS@97** là độ đặc hiệu trung bình mô hình giữ được khi độ nhạy nằm trong
dải 97–100%. Đây **không phải** partial AUC theo chuẩn McClish: phép chuẩn hóa
đó chia cho bề rộng vùng, và bề rộng chính là thứ cần đo. Một mô hình phải trả
nhiều false positive hơn để đạt 97% độ nhạy sẽ vẽ vùng rộng hơn nhưng cùng hình
dạng, và cho điểm y hệt.

Stage B cho thấy vì sao cần bậc này: độ đặc hiệu mức group nhảy theo từng ca
(~0,004), nên gần như mọi khác biệt thật đều rơi vào biên hòa 0,005.

> Đây là quy tắc **v6**. DenseNet v5 dùng NLL làm tie-break đầu tiên, nên so
> sánh DenseNet với DeiT **không phải** một ablation kiến trúc thuần túy.

## Yêu cầu

Bật **Internet** trong Notebook Settings — DeiT cần tải pretrained weights.
Không có nhánh offline, và notebook **dừng** nếu tải hỏng thay vì lặng lẽ
huấn luyện một Transformer khởi tạo ngẫu nhiên.

## Cấu hình

In [1]:
RUN_MODE      = "auto"   # auto | smoke | full
DATA_ROOT_OVERRIDE = None # ví dụ: "/Users/me/data/chest_xray"
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 16
EPOCHS        = 12
LR            = 1e-4
WEIGHT_DECAY  = 1e-5
PATIENCE      = 4         # phải lớn hơn scheduler patience để LR giảm còn có epoch phát huy
SCHEDULER_PATIENCE = 2
SCHEDULER_FACTOR   = 0.3
MIN_LR             = 1e-6
NUM_WORKERS   = 2         # runtime sẽ ép về 0 trên macOS

N_FOLDS       = 5         # 1 = một holdout 15%; 5 = cross-validation đầy đủ
VAL_FRACTION  = 0.15      # chỉ dùng khi N_FOLDS = 1
BORDER_FRAC   = 0.15      # dùng ở phần 4.3
DETERMINISTIC = True      # cudnn tất định; chậm hơn một chút, đổi lại tái lập tốt hơn
RESIZE_MODE   = "letterbox"  # mặc định cho thí nghiệm không ghi rõ "resize"
THRESHOLD_OBJECTIVE = "sensitivity"  # sensitivity | balanced_accuracy
CHECKPOINT_TIE_MARGIN = 0.005  # chênh độ đặc hiệu dưới mức này coi như bằng nhau
TARGET_SENSITIVITY = 0.97
BOOTSTRAP_REPS = 2000    # KTC cho audit tỉ lệ khung ở mức filename-derived group

# Hai bộ augmentation. "mạnh" mô phỏng thiết lập của các cài đặt công khai
# đạt độ đặc hiệu cao hơn: xoay 30 độ, zoom và dịch ảnh.
AUG_PRESETS = {
    "nhe":  {"rotation": 10, "scale": 0.00, "translate": 0.00, "jitter": 0.15},
    "manh": {"rotation": 30, "scale": 0.20, "translate": 0.10, "jitter": 0.20},
}

# Stage A2 chạy đúng một Transformer. Mọi thứ khác giữ nguyên như CNN để
# bảng so sánh còn nghĩa: cùng manifest, cùng fold, cùng preprocessing, cùng loss.
MODEL_ID          = "deit_small_patch16_224.fb_in1k"
REQUIRE_PRETRAINED = True   # tải hỏng thì dừng, không rơi về khởi tạo ngẫu nhiên

DEIT_LR           = 5e-5
DEIT_WEIGHT_DECAY = 0.05
DEIT_BATCH        = 16
WARMUP_EPOCHS     = 1
GRAD_CLIP_NORM    = 1.0

# Biên hòa của quy tắc v6, khóa trước khi chạy.
SPECIFICITY_TIE = 0.005
HSAS_TIE        = 0.002

DEIT_EXPERIMENT = {
    "name": "deit_small", "arch": "deit_small", "size": 224,
    "aug": "manh", "balancing": "weighted", "resize": "stretch",
    "hoi": "Transformer dưới đúng pipeline của CNN",
}
EXPERIMENTS = [DEIT_EXPERIMENT]

# Mốc so sánh, mức filename-group trên known benchmark.
COMPARATORS = {
    "resnet18 B1":  {"auc": 0.9779, "hsas": None, "sensitivity": 0.9951,
                     "specificity": 0.7689, "fp": 52},
    "densenet121":  {"auc": 0.9801, "hsas": None, "sensitivity": 0.9951,
                     "specificity": 0.8044, "fp": 44},
    "ens R+D":      {"auc": 0.9792, "hsas": None, "sensitivity": 0.9951,
                     "specificity": 0.8222, "fp": 40},
}

CLASSES = ("NORMAL", "PNEUMONIA")  # NORMAL=0, PNEUMONIA=1

EXPECTED_STAGE_A1 = {
    "arch": "densenet121", "size": 224, "resize": "stretch",
    "aug": "manh", "balancing": "weighted",
}

In [2]:
import gc, hashlib, json, os, platform, random, re, time, warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import scipy
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = Path("/kaggle/working").is_dir()
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

if RUN_MODE == "auto":
    RUN_MODE = "full" if IS_KAGGLE and DEVICE.type == "cuda" else "smoke"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE phải là 'auto', 'smoke' hoặc 'full'")

if RUN_MODE == "smoke":
    # N_FOLDS giữ nguyên để cách chia khớp các lần chạy CNN.
    EPOCHS, FOLDS_TO_RUN = 1, [0]
else:
    FOLDS_TO_RUN = list(range(N_FOLDS))

if platform.system() == "Darwin":
    NUM_WORKERS = 0  # notebook + spawn không an toàn với closure worker/cache global
LOCAL_PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                           if (p / ".git").is_dir()), Path.cwd())
WORK_DIR = (Path("/kaggle/working") if IS_KAGGLE
            else LOCAL_PROJECT_ROOT / "artifacts/notebook_rerun")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK_DIR / "train_log_deit.txt"
LOG_PATH.write_text("", encoding="utf-8")
PIN_MEMORY = DEVICE.type == "cuda"
AMP_ENABLED = DEVICE.type == "cuda"
DEVICE_NAME = (torch.cuda.get_device_name(0) if DEVICE.type == "cuda"
               else "Apple Metal (MPS)" if DEVICE.type == "mps" else platform.processor() or "CPU")

if DETERMINISTIC and DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_seed(seed=SEED):
    """Seed mọi nguồn ngẫu nhiên mà pipeline đụng tới."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def loader_seed_args(seed=SEED):
    """generator + worker_init_fn cho DataLoader.

    Thiếu hai thứ này thì thứ tự xáo trộn và augmentation chạy trong worker vẫn
    ngẫu nhiên dù đã gọi set_seed — một lỗ hổng tái lập rất hay bị bỏ sót.
    """
    generator = torch.Generator()
    generator.manual_seed(seed)

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        random.seed(worker_seed)
        np.random.seed(worker_seed)

    return {"generator": generator, "worker_init_fn": worker_init_fn}


def log(*parts):
    """In ra màn hình, đồng thời ghi vào LOG_PATH.

    Output của notebook Kaggle có thể mất chunk khi in nhanh; file thì không.
    """
    line = " ".join(str(part) for part in parts)
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")


set_seed()

# Ghi lại phiên bản thư viện. Thiếu nó thì con số trong báo cáo không gắn được
# với môi trường đã sinh ra chúng.
VERSIONS = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pillow": PIL.__version__,
}
with open(WORK_DIR / "environment.json", "w") as handle:
    json.dump({**VERSIONS, "device": str(DEVICE), "device_name": DEVICE_NAME,
               "run_mode": RUN_MODE, "seed": SEED,
               "deterministic": DETERMINISTIC}, handle, indent=2)

# Lưu cấu hình đã resolve sau khi auto/smoke/full được áp dụng. File này giúp
# phân biệt source config với config thực sự sinh ra kết quả.
RESOLVED_CONFIG = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "image_cache_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": DEIT_LR,
    "weight_decay_deit": DEIT_WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "model_id": MODEL_ID,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "scheduler_patience": SCHEDULER_PATIENCE,
    "scheduler_factor": SCHEDULER_FACTOR,
    "min_lr": MIN_LR,
    "checkpoint_tie_margin": CHECKPOINT_TIE_MARGIN,
    "n_folds": N_FOLDS,
    "val_fraction": VAL_FRACTION,
    "deterministic": DETERMINISTIC,
    "threshold_objective": THRESHOLD_OBJECTIVE,
    "target_sensitivity": TARGET_SENSITIVITY,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "augment_presets": AUG_PRESETS,
    "experiments": EXPERIMENTS,
}
with open(WORK_DIR / "resolved_config.json", "w", encoding="utf-8") as handle:
    json.dump(RESOLVED_CONFIG, handle, indent=2, ensure_ascii=False)

log("runtime:", "Kaggle" if IS_KAGGLE else "local", "| mode:", RUN_MODE)
log("device:", DEVICE, f"({DEVICE_NAME})", "| AMP:", AMP_ENABLED,
    "| workers:", NUM_WORKERS)
if RUN_MODE == "smoke":
    log("SMOKE RUN: chỉ kiểm tra pipeline; KHÔNG dùng chỉ số để báo cáo.")
log("phiên bản:", " ".join(f"{k}={v}" for k, v in VERSIONS.items()))

runtime: Kaggle | mode: full
device: cuda (Tesla T4) | AMP: True | workers: 2
phiên bản: python=3.12.13 torch=2.10.0+cu128 torchvision=0.25.0+cu128 numpy=2.0.2 pandas=2.3.3 scikit-learn=1.6.1 scipy=1.16.3 pillow=11.3.0


# 1. Dữ liệu

Giữ nguyên từ v5.

In [3]:
def list_images(directory):
    """Ảnh .jpeg thật, bỏ file sidecar ._* của macOS."""
    return sorted(p for p in Path(directory).glob("*.jpeg")
                  if not p.name.startswith("._"))


def find_data_root(search_paths):
    """Thư mục chứa trực tiếp train/NORMAL và train/PNEUMONIA."""
    if isinstance(search_paths, (str, Path)):
        search_paths = [search_paths]

    candidates = []
    for base in map(Path, search_paths):
        if not base.exists():
            continue
        for train_dir in base.rglob("train"):
            if "__MACOSX" in train_dir.parts:
                continue
            if (train_dir / "NORMAL").is_dir() and (train_dir / "PNEUMONIA").is_dir():
                candidates.append(train_dir.parent.resolve())

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy dataset dưới {[str(p) for p in search_paths]}. "
            "Kaggle: Add Data 'Chest X-Ray Images (Pneumonia)'. "
            "Mac: đặt DATA_ROOT_OVERRIDE hoặc CXR_DATA_ROOT.")

    candidates = sorted(set(candidates), key=lambda path: len(path.parts))
    for candidate in candidates:
        n = len(list_images(candidate / "train" / "NORMAL"))
        mark = "  <- dùng" if candidate == candidates[0] else "  (bản trùng, bỏ qua)"
        print(f"  {candidate}  [{n} ảnh train/NORMAL]{mark}")
    return candidates[0]


explicit_root = DATA_ROOT_OVERRIDE or os.environ.get("CXR_DATA_ROOT")
if explicit_root:
    DATA_ROOT = find_data_root([explicit_root])
else:
    cwd = Path.cwd()
    DATA_ROOT = find_data_root([
        "/kaggle/input", cwd / "chest_xray", cwd.parent / "chest_xray",
        cwd.parent.parent / "chest_xray", cwd / "data/raw",
        cwd.parent / "data/raw",
    ])
log("\nDATA_ROOT =", DATA_ROOT)

  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray  [1341 ảnh train/NORMAL]  <- dùng
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray  [1341 ảnh train/NORMAL]  (bản trùng, bỏ qua)

DATA_ROOT = /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray


In [4]:
PNEUMONIA_RE = re.compile(r"^person(\d+)_(bacteria|virus)_", re.IGNORECASE)
NORMAL_RE    = re.compile(r"^(?:(NORMAL\d+)-)?IM-(\d+)-", re.IGNORECASE)


def parse_group_id(filename):
    """Khoá group suy từ tên file; không khẳng định đây là clinical patient ID."""
    match = PNEUMONIA_RE.match(filename)
    if match:
        return f"pneumonia:{match.group(2).lower()}:{int(match.group(1))}"
    match = NORMAL_RE.match(filename)
    if match:
        return f"normal:{(match.group(1) or 'IM').lower()}:{int(match.group(2))}"
    raise ValueError(f"Tên file lạ, không suy ra được group: {filename}")


def build_manifest(root):
    """Một dòng cho mỗi ảnh: đường dẫn, split gốc, nhãn, filename-derived group."""
    rows = []
    for split in ("train", "val", "test"):
        for class_id, class_name in enumerate(CLASSES):
            directory = Path(root) / split / class_name
            if not directory.is_dir():
                continue
            for path in list_images(directory):
                rows.append({
                    "path": str(path.resolve()), "filename": path.name,
                    "split_original": split, "class_name": class_name,
                    "class_id": class_id, "group_id": parse_group_id(path.name)})
    if not rows:
        raise FileNotFoundError(f"Không có ảnh .jpeg nào dưới {root}")

    frame = pd.DataFrame(rows)
    frame["cache_index"] = np.arange(len(frame))   # vị trí trong cache ở mục 2.2
    return frame


manifest = build_manifest(DATA_ROOT)
log(f"{len(manifest):,} ảnh | {manifest['group_id'].nunique():,} filename-derived groups")
manifest.head(3)

5,856 ảnh | 4,097 filename-derived groups


,path,filename,split_original,class_name,class_id,group_id,cache_index
0,/kaggle/input/datasets/paultimothymooney/chest...,IM-0115-0001.jpeg,train,NORMAL,0,normal:im:115,0
1,/kaggle/input/datasets/paultimothymooney/chest...,IM-0117-0001.jpeg,train,NORMAL,0,normal:im:117,1
2,/kaggle/input/datasets/paultimothymooney/chest...,IM-0119-0001.jpeg,train,NORMAL,0,normal:im:119,2


In [5]:
print("1.3.1  Số lượng theo split và lớp")
print("-" * 62)
print(f"{'split':<8}{'NORMAL':>9}{'PNEUMONIA':>12}{'tổng':>9}{'P/N':>7}")
for split in ("train", "val", "test"):
    subset = manifest[manifest["split_original"] == split]
    counts = subset["class_name"].value_counts()
    normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
    ratio = pneumonia / normal if normal else float("nan")
    print(f"{split:<8}{normal:>9,}{pneumonia:>12,}{normal + pneumonia:>9,}{ratio:>7.2f}")
print(f"{'TỔNG':<8}{'':>9}{'':>12}{len(manifest):>9,}")

1.3.1  Số lượng theo split và lớp
--------------------------------------------------------------
split      NORMAL   PNEUMONIA     tổng    P/N
train       1,341       3,875    5,216   2.89
val             8           8       16   1.00
test          234         390      624   1.67
TỔNG                             5,856


In [6]:
print("1.3.2  Ảnh trùng nội dung (SHA-256)")
print("-" * 62)
manifest["sha256"] = [hashlib.sha256(Path(path).read_bytes()).hexdigest()
                      for path in manifest["path"]]
by_hash = defaultdict(list)
for digest, split in zip(manifest["sha256"], manifest["split_original"]):
    by_hash[digest].append(split)

duplicates = [s for s in by_hash.values() if len(s) > 1]
cross_split = [s for s in duplicates if len(set(s)) > 1]
print(f"tổng file             : {len(manifest):,}")
print(f"hash duy nhất         : {len(by_hash):,}")
print(f"nhóm ảnh trùng        : {len(duplicates)}")
print(f"  trong đó xuyên split: {len(cross_split)}")
print()
print("Không có ảnh y hệt nằm xuyên original split." if not cross_split
      else "CẢNH BÁO: ảnh trùng xuyên original split — kết quả đánh giá bị nhiễm.")

1.3.2  Ảnh trùng nội dung (SHA-256)
--------------------------------------------------------------
tổng file             : 5,856
hash duy nhất         : 5,824
nhóm ảnh trùng        : 30
  trong đó xuyên split: 0

Không có ảnh y hệt nằm xuyên original split.


In [7]:
print("1.3.4  Filename-derived group và nguy cơ trùng giữa các split")
print("-" * 62)
naive_re = re.compile(r"^(person\d+)_", re.IGNORECASE)
naive, corrected = defaultdict(set), defaultdict(set)
for filename, split in zip(manifest["filename"], manifest["split_original"]):
    match = naive_re.match(filename)
    naive[match.group(1).lower() if match else filename].add(split)
    corrected[parse_group_id(filename)].add(split)

naive_span     = sum(1 for s in naive.values() if len(s) > 1)
corrected_span = sum(1 for s in corrected.values() if len(s) > 1)
print(f"khoá person<N>              : {len(naive):,} nhóm, {naive_span} nằm ở >1 split")
print(f"khoá (phân nhóm, person<N>) : {len(corrected):,} nhóm, {corrected_span} nằm ở >1 split")

subtype_ids = defaultdict(set)
for filename in manifest["filename"]:
    match = PNEUMONIA_RE.match(filename)
    if match:
        subtype_ids[match.group(2).lower()].add(int(match.group(1)))
print()
for subtype, ids in sorted(subtype_ids.items()):
    print(f"  {subtype:<9}: {len(ids):,} số, dải 1..{max(ids)}, "
          f"mật độ {len(ids) / max(ids):.3f}")
print(f"  số được dùng bởi CẢ HAI phân nhóm: "
      f"{len(subtype_ids['bacteria'] & subtype_ids['virus']):,}")

per_group = Counter(parse_group_id(f) for f in manifest["filename"])
multi = sum(1 for n in per_group.values() if n > 1)
print(f"\nnhóm có >1 ảnh: {multi:,}/{len(per_group):,} "
      f"(nhiều nhất {max(per_group.values())} ảnh)")

1.3.4  Filename-derived group và nguy cơ trùng giữa các split
--------------------------------------------------------------
khoá person<N>              : 3,257 nhóm, 170 nằm ở >1 split
khoá (phân nhóm, person<N>) : 4,097 nhóm, 0 nằm ở >1 split

  bacteria : 1,437 số, dải 1..1954, mật độ 0.735
  virus    : 1,216 số, dải 1..1685, mật độ 0.722
  số được dùng bởi CẢ HAI phân nhóm: 979

nhóm có >1 ảnh: 726/4,097 (nhiều nhất 30 ảnh)


# 2. Phương pháp

In [8]:
def label_split(manifest, train, val, test):
    out = manifest.copy()
    out["split"] = pd.NA
    out.loc[train.index, "split"] = "train"
    out.loc[val.index,   "split"] = "val"
    out.loc[test.index,  "split"] = "test"
    return out


def make_folds(manifest, n_folds=N_FOLDS, val_fraction=VAL_FRACTION, seed=SEED):
    """Danh sách manifest, mỗi phần tử là một fold đã gán cột split."""
    pool = manifest[manifest["split_original"].isin(["train", "val"])]
    test = manifest[manifest["split_original"] == "test"]
    n_splits = max(2, round(1 / val_fraction)) if n_folds == 1 else n_folds
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds = [label_split(manifest, pool.iloc[train_idx], pool.iloc[val_idx], test)
             for train_idx, val_idx in
             splitter.split(pool, pool["class_id"], groups=pool["group_id"])]
    return folds[:1] if n_folds == 1 else folds


def count_leaked_groups(split):
    """Số filename-derived groups xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("group_id")["split"].nunique() > 1).sum())


def count_leaked_hashes(split):
    """Số nội dung ảnh y hệt xuất hiện ở nhiều split. Phải bằng 0."""
    return int((split.groupby("sha256")["split"].nunique() > 1).sum())


def split_summary(split):
    rows = []
    for name in ("train", "val", "test"):
        subset = split[split["split"] == name]
        counts = subset["class_name"].value_counts()
        normal, pneumonia = int(counts.get("NORMAL", 0)), int(counts.get("PNEUMONIA", 0))
        rows.append({"split": name, "NORMAL": normal, "PNEUMONIA": pneumonia,
                     "tổng": normal + pneumonia,
                     "groups": subset["group_id"].nunique(),
                     "P/N": round(pneumonia / max(normal, 1), 2)})
    return pd.DataFrame(rows).set_index("split")


FOLDS = make_folds(manifest)
log(f"\n{len(FOLDS)} fold, chia theo filename-derived group:")
for i, split in enumerate(FOLDS):
    s = split_summary(split)
    log(f"  fold {i}: train {s.loc['train','tổng']:>5,}  val {s.loc['val','tổng']:>4,}  "
        f"test {s.loc['test','tổng']:>4,}  |  group/hash ở >1 split: "
        f"{count_leaked_groups(split)}/{count_leaked_hashes(split)}")
    assert count_leaked_groups(split) == 0
    assert count_leaked_hashes(split) == 0
    split.to_csv(WORK_DIR / f"manifest_fold{i}.csv", index=False)

print("\nChi tiết fold 0:")
display(split_summary(FOLDS[0]))


5 fold, chia theo filename-derived group:
  fold 0: train 4,168  val 1,064  test  624  |  group/hash ở >1 split: 0/0
  fold 1: train 4,224  val 1,008  test  624  |  group/hash ở >1 split: 0/0
  fold 2: train 4,165  val 1,067  test  624  |  group/hash ở >1 split: 0/0
  fold 3: train 4,190  val 1,042  test  624  |  group/hash ở >1 split: 0/0
  fold 4: train 4,181  val 1,051  test  624  |  group/hash ở >1 split: 0/0

Chi tiết fold 0:


,NORMAL,PNEUMONIA,tổng,groups,P/N
split,,,,,
train,1092,3076,4168,2934,2.82
val,257,807,1064,735,3.14
test,234,390,624,428,1.67


## 2.2. Tiền xử lý và augmentation

In [9]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)


def device_augment(batch, cfg):
    """Lật, xoay, zoom, dịch, đổi sáng/tương phản trên CUDA/MPS/CPU.

    Làm bằng PIL trong DataLoader thì CPU thành nút thắt: riêng RandomRotation
    và ColorJitter đã ngốn hơn 1.000 lần thời gian đọc cache. Ở đây mọi phép
    biến đổi là tensor op chạy theo lô trên accelerator đang chọn.

    Xoay, zoom và dịch được gộp vào MỘT phép biến đổi affine, nên chỉ nội suy
    một lần thay vì ba lần chồng lên nhau.
    """
    n, dev = batch.size(0), batch.device
    flip = torch.rand(n, device=dev) < 0.5
    batch = torch.where(flip.view(-1, 1, 1, 1), batch.flip(-1), batch)

    rand = lambda: torch.rand(n, device=dev) * 2 - 1          # noqa: E731  -1..1
    angles = rand() * (cfg["rotation"] * np.pi / 180)
    zoom = 1 + rand() * cfg["scale"]
    cos, sin = torch.cos(angles) * zoom, torch.sin(angles) * zoom
    theta = torch.zeros(n, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = rand() * cfg["translate"]
    theta[:, 1, 2] = rand() * cfg["translate"]
    grid = F.affine_grid(theta, batch.shape, align_corners=False)
    batch = F.grid_sample(batch, grid, align_corners=False, padding_mode="zeros")

    j = cfg["jitter"]
    scale = 1 + rand().view(-1, 1, 1, 1) * j
    contrast = 1 + rand().view(-1, 1, 1, 1) * j
    mean = batch.mean(dim=(1, 2, 3), keepdim=True)
    return ((batch * scale - mean) * contrast + mean).clamp(0, 1)


def to_model_input(batch_uint8, size=IMG_SIZE, aug=None):
    """(B,H,W) uint8 -> (B,3,H,W) chuẩn hoá trên DEVICE.

    Cache giữ ảnh ở IMG_SIZE; thí nghiệm nào cần kích thước khác thì thu nhỏ
    ngay trên DEVICE. Đây là resize hai bước (gốc -> IMG_SIZE -> size), áp dụng
    đồng nhất cho mọi split nên không tạo chênh lệch giữa train và test.
    """
    x = batch_uint8.to(DEVICE, non_blocking=True).float().div_(255).unsqueeze(1)
    if size != IMG_SIZE:
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    if aug is not None:
        x = device_augment(x, aug)
    return (x.expand(-1, 3, -1, -1) - MEAN_T) / STD_T


def resize_for_cache(image, size=IMG_SIZE, mode=RESIZE_MODE):
    gray = image.convert("L")
    if mode == "stretch":
        return np.asarray(gray.resize((size, size), Image.Resampling.BILINEAR))
    if mode != "letterbox":
        raise ValueError(f"RESIZE_MODE lạ: {mode}")
    gray.thumbnail((size, size), Image.Resampling.BILINEAR)
    array = np.asarray(gray)
    fill = int(np.median(array))
    canvas = Image.new("L", (size, size), color=fill)
    offset = ((size - gray.width) // 2, (size - gray.height) // 2)
    canvas.paste(gray, offset)
    return np.asarray(canvas)


def build_image_cache(manifest, size=IMG_SIZE, mode=RESIZE_MODE):
    cache = np.zeros((len(manifest), size, size), dtype=np.uint8)
    for position, path in enumerate(manifest["path"]):
        with Image.open(path) as image:
            cache[position] = resize_for_cache(image, size, mode)
        if (position + 1) % 1500 == 0:
            print(f"  {position + 1:,}/{len(manifest):,}")
    return cache


# Mỗi chế độ resize cần một cache riêng. Chỉ dựng những chế độ thực sự được
# dùng, để so sánh stretch với letterbox nằm trong cùng một lần chạy thay vì hai
# lần chạy khác nhau như trước.
REQUIRED_MODES = sorted({spec.get("resize", RESIZE_MODE) for spec in EXPERIMENTS})
IMAGE_CACHES = {}
for mode in REQUIRED_MODES:
    _started = time.time()
    IMAGE_CACHES[mode] = build_image_cache(manifest, mode=mode)
    log(f"cache {mode}: {IMAGE_CACHES[mode].nbytes / 1e6:.0f} MB cho "
        f"{len(manifest):,} ảnh trong {time.time() - _started:.0f}s")

# Cache mặc định cho các đoạn không gắn với một thí nghiệm cụ thể.
IMAGE_CACHE = IMAGE_CACHES[RESIZE_MODE if RESIZE_MODE in IMAGE_CACHES
                           else REQUIRED_MODES[0]]


class XRayDataset(Dataset):
    """Trả về ảnh uint8 thô; augmentation diễn ra trên DEVICE."""

    def __init__(self, rows, mode=RESIZE_MODE):
        self.rows, self.mode = rows, mode

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        cache_index, label = self.rows[index]
        return torch.from_numpy(IMAGE_CACHES[self.mode][cache_index]), label


def make_loader(split, name, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    subset = split[split["split"] == name]
    return DataLoader(
        XRayDataset(list(zip(subset["cache_index"], subset["class_id"])), mode),
        batch_size=batch_size, shuffle=(name == "train"),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
        **loader_seed_args(seed))


def make_loaders(split, batch_size=BATCH_SIZE, seed=SEED, mode=RESIZE_MODE):
    return {name: make_loader(split, name, batch_size, seed, mode)
            for name in ("train", "val")}


def class_weights_from(split):
    """Trọng số nghịch tần suất, chuẩn hoá để loss giữ nguyên thang đo."""
    counts = Counter(split[split["split"] == "train"]["class_id"])
    total = sum(counts.values())
    return torch.tensor([total / (len(CLASSES) * counts[i]) for i in range(len(CLASSES))],
                        dtype=torch.float, device=DEVICE)

  1,500/5,856
  3,000/5,856
  4,500/5,856
cache stretch: 294 MB cho 5,856 ảnh trong 57s


## 2.3. Chỉ số đánh giá

In [10]:
METRIC_COLS = ["accuracy", "precision", "recall", "specificity",
               "f1", "bal_acc", "auc", "pr_auc"]


def metrics_at(labels, probs, threshold=0.5):
    """Chấm điểm tại một ngưỡng. threshold=0.5 chính là argmax trên hai logit."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sensitivity, specificity = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    return {
        "threshold": float(threshold),
        "accuracy": float((labels == preds).mean()),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "specificity": float(specificity),
        "f1": f1_score(labels, preds, zero_division=0),
        "bal_acc": float((sensitivity + specificity) / 2),
        "auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
    }


def to_group_level(group_ids, labels, probs):
    """Gộp theo filename-derived group; xác suất là trung bình các ảnh."""
    frame = pd.DataFrame({"group": group_ids, "label": labels, "prob": probs})
    label_counts = frame.groupby("group")["label"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("group", sort=True).agg(
        label=("label", "first"), prob=("prob", "mean"))
    return rolled["label"].to_numpy(), rolled["prob"].to_numpy()


def tune_threshold(labels, probs, objective=THRESHOLD_OBJECTIVE,
                   target_sensitivity=TARGET_SENSITIVITY):
    """Chọn một candidate thật trên validation/OOF, không nội suy qua vùng tie."""
    labels, probs = np.asarray(labels), np.asarray(probs)
    candidates = np.unique(np.clip(probs, 0.001, 0.999))
    if len(candidates) > 400:
        indices = np.linspace(0, len(candidates) - 1, 400).round().astype(int)
        candidates = candidates[np.unique(indices)]

    rows = [metrics_at(labels, probs, float(t)) for t in candidates]
    if objective == "balanced_accuracy":
        best = max(rows, key=lambda m: (m["bal_acc"], m["specificity"],
                                        m["recall"], m["threshold"]))
    elif objective == "sensitivity":
        feasible = [m for m in rows if m["recall"] >= target_sensitivity - 1e-12]
        if not feasible:
            warnings.warn("Không có ngưỡng đạt target sensitivity; dùng recall cao nhất.")
            feasible = rows
            best = max(feasible, key=lambda m: (m["recall"], m["specificity"],
                                                m["threshold"]))
        else:
            best = max(feasible, key=lambda m: (m["specificity"], m["bal_acc"],
                                                m["threshold"]))
    else:
        raise ValueError(f"THRESHOLD_OBJECTIVE lạ: {objective}")
    return float(best["threshold"]), best

## 2.4. Kiểm tra model trước khi chạy

Xác nhận model ID tồn tại, weights tải được, output đúng hình dạng, và chuẩn
hóa của model khớp cái pipeline đang dùng. Hash riêng backbone — head hai lớp
khởi tạo mới mỗi lần nên đưa vào sẽ làm digest đổi liên tục.

In [11]:
import timm

# Fail here rather than at the first forward pass, and never quietly train a
# randomly initialised Transformer: it would look like a weak architecture
# result instead of a missing download.
available = timm.list_models(f"*{MODEL_ID.split('.')[0]}*", pretrained=True)
if MODEL_ID not in available:
    raise RuntimeError(
        f"{MODEL_ID} không có trong timm {timm.__version__}. "
        f"Gần nhất: {available[:5]}")

set_seed()  # trước create_model, để head 2 lớp khởi tạo tái lập được
try:
    _probe = timm.create_model(MODEL_ID, pretrained=REQUIRE_PRETRAINED,
                               num_classes=len(CLASSES))
except Exception as exc:
    raise RuntimeError(
        "Không tải được pretrained weights. Bật Internet trong Notebook "
        "Settings. Không fallback sang khởi tạo ngẫu nhiên."
    ) from exc

_cfg = _probe.default_cfg
DEIT_PROVENANCE = {
    "timm_version": timm.__version__, "model_id": MODEL_ID,
    "pretrained": REQUIRE_PRETRAINED, "num_classes": len(CLASSES),
    "input_size": list(_cfg["input_size"]),
    "mean": list(_cfg["mean"]), "std": list(_cfg["std"]),
    "parameters_millions": sum(p.numel() for p in _probe.parameters()) / 1e6,
}

# Hash the backbone only. The classification head is freshly initialised for
# two classes, so including it would make the digest change every run.
_backbone = {k: v for k, v in _probe.state_dict().items()
             if not k.startswith("head")}
DEIT_PROVENANCE["backbone_sha256"] = hashlib.sha256(
    b"".join(v.cpu().numpy().tobytes() for _, v in sorted(_backbone.items()))
).hexdigest()[:16]

with open(WORK_DIR / "checkpoint_manifest_deit.json", "w") as handle:
    json.dump(DEIT_PROVENANCE, handle, indent=2)

log(f"model: {MODEL_ID}")
log(f"timm: {timm.__version__} | pretrained: {REQUIRE_PRETRAINED} | "
    f"tham số: {DEIT_PROVENANCE['parameters_millions']:.1f}M")
log(f"backbone sha256: {DEIT_PROVENANCE['backbone_sha256']}")

_out = _probe(torch.randn(2, 3, 224, 224))
assert _out.shape == (2, len(CLASSES)), f"output shape lạ: {tuple(_out.shape)}"
log(f"output shape: {tuple(_out.shape)}")

# The pipeline already normalises with these values; a mismatch would mean the
# cache and the pretrained weights disagree about what the input looks like.
assert np.allclose(_cfg["mean"], IMAGENET_MEAN), "mean của model khác pipeline"
assert np.allclose(_cfg["std"], IMAGENET_STD), "std của model khác pipeline"
log("chuẩn hóa khớp pipeline hiện tại")
del _probe, _out
gc.collect()

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

model: deit_small_patch16_224.fb_in1k
timm: 1.0.26 | pretrained: True | tham số: 21.7M
backbone sha256: cb489a7792932483
output shape: (2, 2)
chuẩn hóa khớp pipeline hiện tại


83

## 2.5. Hàm dùng chung từ v5

In [12]:
@torch.no_grad()
def predict(model, loader, size=IMG_SIZE):
    """(nhãn thật, xác suất PNEUMONIA) trên toàn bộ loader."""
    model.eval()
    labels_all, probs_all = [], []
    for images, labels in loader:
        logits = model(to_model_input(images, size))
        probs_all += torch.softmax(logits.float(), dim=1)[:, 1].cpu().tolist()
        labels_all += labels.tolist()
    return np.array(labels_all), np.array(probs_all)


def group_scores(labels, probs, groups):
    """Gộp dự đoán về mức filename-derived group."""
    frame = pd.DataFrame({"g": groups, "y": labels, "p": probs})
    label_counts = frame.groupby("g")["y"].nunique()
    if int(label_counts.max()) != 1:
        bad = label_counts[label_counts > 1].index.tolist()[:5]
        raise ValueError(f"Group validation chứa nhiều nhãn, ví dụ: {bad}")
    rolled = frame.groupby("g", sort=True).agg(y=("y", "first"), p=("p", "mean"))
    return rolled["y"].to_numpy(), rolled["p"].to_numpy()


def specificity_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Độ đặc hiệu cao nhất còn giữ được độ nhạy tối thiểu, kèm ngưỡng.

    Đây là chỉ số dự án đang thực sự cần cải thiện. Validation AUC đã bão hòa ở
    0,999x nên chọn epoch theo nó chỉ là chọn theo nhiễu.
    """
    positive = probs[labels == 1]
    if not len(positive):
        return 0.0, 0.5
    feasible = [c for c in np.unique(probs) if (positive >= c).mean() >= target]
    if not feasible:
        return 0.0, 0.0
    threshold = float(max(feasible))
    return float((probs[labels == 0] < threshold).mean()), threshold


def group_nll(labels, probs):
    """Log-loss không trọng số ở mức group.

    Không dùng loss có trọng số lớp để đánh giá: trọng số làm lệch thang xác
    suất, nên nó không nói được mô hình hiệu chuẩn tốt hay xấu.
    """
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def group_brier(labels, probs):
    """Brier score ở mức group."""
    return float(np.mean((probs - labels) ** 2))


def better_checkpoint(candidate, incumbent, margin=CHECKPOINT_TIE_MARGIN):
    """Quy tắc chọn checkpoint, khóa trước khi chạy.

    Độ đặc hiệu là chỉ số chính. Chênh lệch dưới ``margin`` coi như bằng nhau và
    NLL quyết định, vì độ đặc hiệu ở mức group nhảy bậc rời rạc — một ca đổi
    phía đã là 0,004 — nên chênh lệch nhỏ hơn thế là nhiễu lấy mẫu.

    Args:
        candidate: Chỉ số của epoch hiện tại.
        incumbent: Chỉ số của checkpoint đang giữ, hoặc None.
        margin: Ngưỡng coi hai độ đặc hiệu là bằng nhau.

    Returns:
        True nếu nên thay checkpoint.
    """
    if incumbent is None:
        return True
    gap = candidate["specificity"] - incumbent["specificity"]
    if gap > margin + 1e-12:
        return True
    if gap < -margin - 1e-12:
        return False
    # Hòa về độ đặc hiệu: lấy NLL thấp hơn. Vẫn hòa thì giữ epoch sớm hơn.
    return candidate["nll"] < incumbent["nll"] - 1e-9

## 2.6. Chỉ số và quy tắc chọn checkpoint v6

Định nghĩa sau các hàm của v5 nên chúng ghi đè bản cũ: v5 chọn checkpoint bằng
`spec → NLL`, v6 chèn HSAS@97 vào giữa và trả về cả lý do.

In [13]:
def exact_threshold_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Highest observed score that still meets a minimum sensitivity.

    Every distinct probability is a candidate. At group level there are only a
    few thousand, and a fixed grid can step straight over the one value that
    separates two cases.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        target: Minimum sensitivity to hold.

    Returns:
        The selected threshold, or 0.0 when the target is unreachable.
    """
    labels, probs = np.asarray(labels), np.asarray(probs, dtype=float)
    positive = probs[labels == 1]
    if not len(positive):
        return 0.5
    candidates = np.unique(probs)
    feasible = candidates[[(positive >= c).mean() >= target for c in candidates]]
    return float(feasible.max()) if len(feasible) else 0.0


def specificity_at_sensitivity(labels, probs, target=TARGET_SENSITIVITY):
    """Best specificity reachable while holding a minimum sensitivity.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        target: Minimum sensitivity to hold.

    Returns:
        Tuple of (specificity, threshold).
    """
    threshold = exact_threshold_at_sensitivity(labels, probs, target)
    negative = np.asarray(probs, dtype=float)[np.asarray(labels) == 0]
    if not len(negative):
        return 0.0, threshold
    return float((negative < threshold).mean()), threshold


def hsas_97(labels, probs, min_sensitivity=TARGET_SENSITIVITY):
    """Mean specificity held across sensitivities from the target to 1.

    Reported as HSAS@97, and deliberately not called partial AUC: the McClish
    normalisation divides by the region's own width, which cancels exactly what
    matters here. A model paying far more false positives to reach 97%
    sensitivity traces a wider region of the same shape and would score the
    same.

    Group specificity moves in steps of about one case, so nearly every real
    difference between epochs lands inside the tie band. This reads the same
    part of the curve continuously and can separate them.

    Args:
        labels: Binary labels.
        probs: Predicted probabilities.
        min_sensitivity: Lower bound of the sensitivity range.

    Returns:
        Mean specificity over the range, in [0, 1].

    Raises:
        ValueError: If either class is missing.
    """
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2:
        raise ValueError(f"HSAS cần cả hai lớp; thấy {np.unique(labels).tolist()}")
    fpr, tpr, _ = roc_curve(labels, np.asarray(probs, dtype=float),
                            drop_intermediate=False)
    # Several thresholds can reach one sensitivity; only the cheapest is a
    # real operating point. Interpolating through the others charges false
    # positives the model never had to pay.
    order = np.lexsort((fpr, tpr))
    tpr, fpr = tpr[order], fpr[order]
    keep = np.r_[True, np.diff(tpr) > 0]
    tpr, fpr = tpr[keep], fpr[keep]
    assert np.all(np.diff(fpr) >= -1e-12), "biên Pareto không đơn điệu"
    grid = np.linspace(min_sensitivity, 1.0, 512)
    return float(np.trapezoid(1.0 - np.interp(grid, tpr, fpr), grid)
                 / (1.0 - min_sensitivity))


def better_checkpoint(candidate, incumbent):
    """Decide whether an epoch replaces the one currently held.

    Specificity decides first because it is the endpoint. Inside its tie band
    HSAS@97 decides, then log-loss, then the earlier epoch. The reason is
    returned so the epoch history records why, rather than leaving it to be
    reconstructed from the numbers.

    Args:
        candidate: Metrics for the current epoch.
        incumbent: Metrics for the held checkpoint, or None.

    Returns:
        Tuple of (replace, reason).
    """
    if incumbent is None:
        return True, "first"
    gap = candidate["specificity"] - incumbent["specificity"]
    if gap > SPECIFICITY_TIE:
        return True, "higher_specificity"
    if gap < -SPECIFICITY_TIE:
        return False, "lower_specificity"

    difference = candidate["hsas_97"] - incumbent["hsas_97"]
    if difference > HSAS_TIE:
        return True, "specificity_tie_higher_hsas"
    if difference < -HSAS_TIE:
        return False, "specificity_tie_lower_hsas"

    if candidate["nll"] < incumbent["nll"] - 1e-9:
        return True, "specificity_hsas_tie_lower_nll"
    return False, "all_tied_keep_earlier"


def group_scores(labels, probs, groups):
    """Collapse image predictions to one score per filename-derived group."""
    frame = pd.DataFrame({"g": groups, "y": labels, "p": probs})
    rolled = frame.groupby("g").agg(y=("y", "first"), p=("p", "mean"))
    return rolled["y"].to_numpy(), rolled["p"].to_numpy()


def group_nll(labels, probs):
    """Unweighted log-loss at group level.

    Unweighted on purpose: class weights distort the probability scale, so a
    weighted loss cannot report whether the model is calibrated.
    """
    p = np.clip(probs, 1e-7, 1 - 1e-7)
    return float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p)))


def group_brier(labels, probs):
    """Brier score at group level."""
    return float(np.mean((probs - labels) ** 2))

## 2.7. Vòng huấn luyện

Warm-up rồi cosine decay, tính theo **lần cập nhật optimizer** chứ không theo
mini-batch vật lý — nếu batch 16 không vừa GPU thì batch 8 với accumulation 2
vẫn cho đúng một lịch learning rate.

Gradient được unscale trước khi clip. Clip trên gradient còn scale sẽ áp một
ngưỡng phụ thuộc loss scale, tức là một ngưỡng khác ở mỗi bước.

In [14]:
def build_deit():
    """Create a pretrained DeiT-Small with a two-class head.

    Returns:
        The model on DEVICE.
    """
    model = timm.create_model(MODEL_ID, pretrained=REQUIRE_PRETRAINED,
                              num_classes=len(CLASSES))
    return model.to(DEVICE)


def resolve_batch_size(model, size, requested=DEIT_BATCH):
    """Find a physical batch that fits, and accumulate to keep the effective one.

    Halving the batch without accumulating would change the optimisation
    problem, not just the memory footprint, so the two must move together.

    Args:
        model: The model to test.
        size: Input side length.
        requested: Desired effective batch size.

    Returns:
        Tuple of (physical batch, accumulation steps).
    """
    if DEVICE.type != "cuda":
        return requested, 1
    for physical in (requested, requested // 2):
        try:
            torch.cuda.empty_cache()
            probe = torch.zeros(physical, size, size, dtype=torch.uint8)
            with torch.amp.autocast(device_type="cuda", enabled=AMP_ENABLED):
                loss = model(to_model_input(probe, size)).float().sum()
            loss.backward()
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            return physical, requested // physical
        except torch.cuda.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    raise RuntimeError("Không vừa cả batch 16 lẫn 8 trên GPU này")


def run_fold(spec, fold_index, epochs=EPOCHS):
    """Train one fold of DeiT-Small.

    Args:
        spec: Experiment specification.
        fold_index: Which fold to run.
        epochs: Maximum epochs.

    Returns:
        Result mapping matching the shape the CNN runs produced.
    """
    log(f"\n{'=' * 62}\n{spec['name']}  |  fold {fold_index}\n{'=' * 62}")
    resize, size = spec["resize"], spec["size"]
    aug = AUG_PRESETS[spec["aug"]]
    set_seed(SEED + fold_index)

    split = FOLDS[fold_index]
    model = build_deit()
    physical, accumulation = resolve_batch_size(model, size)
    loaders = make_loaders(split, seed=SEED + fold_index, mode=resize,
                           batch_size=physical)
    log(f"{MODEL_ID} | {size}px | resize {resize} | augment {spec['aug']} | "
        f"balancing {spec['balancing']}")
    log(f"AdamW | lr {DEIT_LR:.0e} | wd {DEIT_WEIGHT_DECAY} | "
        f"batch {physical}×{accumulation} = {physical * accumulation}")
    log("quy tắc chọn: spec@sens97 → HSAS@97 → NLL → epoch sớm hơn")

    weights = (class_weights_from(split) if spec["balancing"] == "weighted"
               else None)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=DEIT_LR,
                                  weight_decay=DEIT_WEIGHT_DECAY)
    steps_per_epoch = max(len(loaders["train"]) // accumulation, 1)
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    total_steps = epochs * steps_per_epoch

    def lr_scale(step):
        """Linear warm-up then cosine decay, indexed by optimizer updates."""
        if step < warmup_steps:
            return (step + 1) / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + np.cos(np.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_scale)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    val_rows = split[split["split"] == "val"].reset_index(drop=True)
    val_groups = val_rows["group_id"].to_numpy()

    best, best_epoch, best_state, stale, history = None, 0, None, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        running, seen = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (images, labels) in enumerate(loaders["train"]):
            inputs = to_model_input(images, size, aug)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss = criterion(model(inputs), labels) / accumulation
            scaler.scale(loss).backward()
            running += loss.item() * accumulation * inputs.size(0)
            seen += inputs.size(0)

            if (step + 1) % accumulation == 0 or step + 1 == len(loaders["train"]):
                # Unscale before clipping: clipping scaled gradients would
                # apply a threshold that depends on the loss scale.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),
                                               GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        labels, probs = predict(model, loaders["val"], size)
        g_labels, g_probs = group_scores(labels, probs, val_groups)
        specificity, threshold = specificity_at_sensitivity(g_labels, g_probs)
        operating = metrics_at(g_labels, g_probs, threshold)
        (tn, fp), (fn, tp) = operating["confusion_matrix"]
        current = {
            "epoch": epoch, "train_loss": running / seen,
            "specificity": specificity, "sensitivity": operating["recall"],
            "hsas_97": hsas_97(g_labels, g_probs), "threshold": threshold,
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
            "nll": group_nll(g_labels, g_probs),
            "brier": group_brier(g_labels, g_probs),
            "auc": roc_auc_score(g_labels, g_probs),
            "pr_auc": average_precision_score(g_labels, g_probs),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        replace, reason = better_checkpoint(current, best)
        current["selected"] = replace
        current["selection_reason"] = reason
        history.append(current)

        if replace:
            best, best_epoch, stale = current, epoch, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            stale += 1

        log(f"epoch {epoch:>2}/{epochs}  loss {current['train_loss']:.4f}  "
            f"spec@sens{TARGET_SENSITIVITY:.0%} {specificity:.4f}  "
            f"HSAS {current['hsas_97']:.4f}  AUC {current['auc']:.4f}  "
            f"NLL {current['nll']:.4f}  lr {current['learning_rate']:.2e}  "
            f"{reason}{'  <- best' if replace else ''}")
        if stale >= PATIENCE:
            log(f"dừng sớm ở epoch {epoch}")
            break

    model.load_state_dict(best_state)
    log(f"giữ checkpoint epoch {best_epoch} ({best['selection_reason']}), "
        f"spec {best['specificity']:.4f}, HSAS {best['hsas_97']:.4f}")

    peak = {}
    if DEVICE.type == "cuda":
        peak = {"peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9,
                "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1e9}
        log(f"GPU đỉnh: cấp phát {peak['peak_allocated_gb']:.2f} GB, "
            f"dành riêng {peak['peak_reserved_gb']:.2f} GB")
        torch.cuda.reset_peak_memory_stats()

    val_labels, val_probs = predict(model, loaders["val"], size)
    tag = f"{spec['name']}_fold{fold_index}"
    torch.save(best_state, WORK_DIR / f"{tag}.pth")
    val_rows.assign(p_pneumonia=val_probs).to_csv(
        WORK_DIR / f"validation_predictions_{tag}.csv", index=False)
    pd.DataFrame(history).to_csv(WORK_DIR / f"epoch_history_{tag}.csv",
                                 index=False)

    result = {"experiment": spec["name"], "resize": resize, "arch": spec["arch"],
              "size": size, "balancing": spec["balancing"], "fold": fold_index,
              "best_epoch": best_epoch, "checkpoint": str(WORK_DIR / f"{tag}.pth"),
              "physical_batch": physical, "accumulation": accumulation,
              "val_labels": val_labels, "val_probs": val_probs,
              "val_groups": val_groups,
              "val": metrics_at(val_labels, val_probs, 0.5),
              "selection": best, **peak}
    del model, loaders, optimizer, scheduler, scaler, best_state
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()
    return result

# 3. Kết quả

In [15]:
log(f"Bắt đầu {RUN_MODE}: {len(EXPERIMENTS)} thí nghiệm × {len(FOLDS)} fold × "
    f"tối đa {EPOCHS} epoch")
_t0 = time.time()
RUNS = [run_fold(spec, fold)
        for spec in EXPERIMENTS
        for fold in FOLDS_TO_RUN]
log(f"\ntổng thời gian: {(time.time() - _t0) / 60:.1f} phút "
    f"({len(EXPERIMENTS)} thí nghiệm × {len(FOLDS_TO_RUN)} fold)")

Bắt đầu full: 1 thí nghiệm × 5 fold × tối đa 12 epoch

deit_small  |  fold 0
deit_small_patch16_224.fb_in1k | 224px | resize stretch | augment manh | balancing weighted
AdamW | lr 5e-05 | wd 0.05 | batch 16×1 = 16
quy tắc chọn: spec@sens97 → HSAS@97 → NLL → epoch sớm hơn
epoch  1/12  loss 0.2878  spec@sens97% 0.9506  HSAS 0.8433  AUC 0.9933  NLL 0.1319  lr 5.00e-05  first  <- best
epoch  2/12  loss 0.1594  spec@sens97% 0.9547  HSAS 0.8348  AUC 0.9923  NLL 0.2335  lr 4.90e-05  specificity_tie_lower_hsas
epoch  3/12  loss 0.1268  spec@sens97% 0.9753  HSAS 0.8167  AUC 0.9926  NLL 0.1536  lr 4.60e-05  higher_specificity  <- best
epoch  4/12  loss 0.1012  spec@sens97% 0.9671  HSAS 0.8961  AUC 0.9958  NLL 0.1579  lr 4.14e-05  lower_specificity
epoch  5/12  loss 0.0952  spec@sens97% 0.9465  HSAS 0.7907  AUC 0.9917  NLL 0.3603  lr 3.54e-05  lower_specificity
epoch  6/12  loss 0.0716  spec@sens97% 0.9753  HSAS 0.9087  AUC 0.9962  NLL 0.1386  lr 2.86e-05  specificity_tie_higher_hsas  <- best
epo

In [16]:
display(pd.DataFrame([
    {"fold": r["fold"], "epoch": r["best_epoch"],
     "lý do": r["selection"]["selection_reason"],
     "spec@sens97": round(r["selection"]["specificity"], 4),
     "HSAS@97": round(r["selection"]["hsas_97"], 4),
     "NLL": round(r["selection"]["nll"], 4),
     "batch": f"{r['physical_batch']}×{r['accumulation']}"}
    for r in RUNS]).set_index("fold"))

if any("peak_allocated_gb" in r for r in RUNS):
    print(f"\nGPU đỉnh qua các fold: "
          f"cấp phát {max(r.get('peak_allocated_gb', 0) for r in RUNS):.2f} GB, "
          f"dành riêng {max(r.get('peak_reserved_gb', 0) for r in RUNS):.2f} GB")

print("\nBenchmark chưa được đọc ở bước này.")

,epoch,lý do,spec@sens97,HSAS@97,NLL,batch
fold,,,,,,
0,9,higher_specificity,0.9959,0.9364,0.2082,16×1
1,5,specificity_tie_higher_hsas,0.9918,0.9770,0.0959,16×1
2,4,higher_specificity,0.9754,0.9387,0.1550,16×1
3,7,higher_specificity,0.9836,0.9352,0.1388,16×1
4,9,higher_specificity,0.9959,0.9644,0.0759,16×1



GPU đỉnh qua các fold: cấp phát 0.91 GB, dành riêng 1.01 GB

Benchmark chưa được đọc ở bước này.


## 3.2. Tổng hợp OOF và khóa ngưỡng

In [17]:
oof_labels = np.concatenate([r["val_labels"] for r in RUNS])
oof_probs = np.concatenate([r["val_probs"] for r in RUNS])
oof_groups = np.concatenate([r["val_groups"] for r in RUNS])
G_LABELS, G_PROBS = group_scores(oof_labels, oof_probs, oof_groups)

OOF_THRESHOLD = exact_threshold_at_sensitivity(G_LABELS, G_PROBS)
OOF_SPECIFICITY, _ = specificity_at_sensitivity(G_LABELS, G_PROBS)
OOF_HSAS = hsas_97(G_LABELS, G_PROBS)

pd.DataFrame([{"experiment": "deit_small", "unit": "filename_group",
               "n_groups": len(G_LABELS), "threshold": OOF_THRESHOLD,
               "specificity_at_sens97": OOF_SPECIFICITY, "hsas_97": OOF_HSAS,
               "auc": roc_auc_score(G_LABELS, G_PROBS),
               "pr_auc": average_precision_score(G_LABELS, G_PROBS),
               "nll": group_nll(G_LABELS, G_PROBS),
               "brier": group_brier(G_LABELS, G_PROBS)}]
             ).to_csv(WORK_DIR / "results_deit_oof.csv", index=False)

print(f"OOF gộp: {len(G_LABELS):,} group")
print(f"  ngưỡng khóa        : {OOF_THRESHOLD:.6f}")
print(f"  spec@sens97        : {OOF_SPECIFICITY:.4f}")
print(f"  HSAS@97            : {OOF_HSAS:.4f}")

# Cổng một chiều: benchmark chỉ được đọc sau dòng này.
OOF_THRESHOLD_LOCKED = True
print("\nOOF_THRESHOLD_LOCKED = True")

OOF gộp: 3,669 group
  ngưỡng khóa        : 0.106037
  spec@sens97        : 0.9852
  HSAS@97            : 0.9101

OOF_THRESHOLD_LOCKED = True


## 3.3. Known benchmark

Chỉ chạy sau khi ngưỡng đã khóa bằng OOF.

In [18]:
assert OOF_THRESHOLD_LOCKED, "Chưa khóa ngưỡng OOF — không được đọc benchmark"

test_rows = FOLDS[0][FOLDS[0]["split"] == "test"].reset_index(drop=True)
test_loader = make_loader(FOLDS[0], "test", seed=SEED,
                          mode=DEIT_EXPERIMENT["resize"])

stack, test_labels = [], None
for run in sorted(RUNS, key=lambda r: r["fold"]):
    model = build_deit()
    model.load_state_dict(torch.load(run["checkpoint"], map_location="cpu",
                                     weights_only=True))
    model.eval()
    labels, probs = predict(model, test_loader, run["size"])
    test_labels = labels if test_labels is None else test_labels
    assert np.array_equal(test_labels, labels)
    stack.append(probs)
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

test_probs = np.mean(stack, axis=0)
B_LABELS, B_PROBS = group_scores(test_labels, test_probs,
                                 test_rows["group_id"].to_numpy())
block = metrics_at(B_LABELS, B_PROBS, OOF_THRESHOLD)
(tn, fp), (fn, tp) = block["confusion_matrix"]

test_rows.assign(p_pneumonia=test_probs,
                 pred=(test_probs >= OOF_THRESHOLD).astype(int)).to_csv(
    WORK_DIR / "predictions_known_benchmark_deit_small_images.csv", index=False)
pd.DataFrame({"group_id": np.unique(test_rows["group_id"]),
              "label": B_LABELS, "p_pneumonia": B_PROBS,
              "pred": (B_PROBS >= OOF_THRESHOLD).astype(int)}).to_csv(
    WORK_DIR / "predictions_known_benchmark_deit_small_groups.csv", index=False)

rows = [{"model": name, **stats} for name, stats in COMPARATORS.items()]
rows.append({"model": "deit_small", "auc": block["auc"],
             "hsas": hsas_97(B_LABELS, B_PROBS),
             "sensitivity": block["recall"], "specificity": block["specificity"],
             "fp": int(fp)})
comparison = pd.DataFrame(rows)
comparison.to_csv(WORK_DIR / "results_deit_small.csv", index=False)
display(comparison.round(4))

print(f"\nTN {tn}  FP {fp}  FN {fn}  TP {tp}")
print(f"khoảng cách OOF→benchmark, độ đặc hiệu: "
      f"{OOF_SPECIFICITY - block['specificity']:+.4f}")

keep = (block["recall"] >= 0.97
        and (block["specificity"] >= 0.82
             or hsas_97(B_LABELS, B_PROBS) >= max(0.8766, 0.8596) + 0.01))
print(f"\n=> {'GIỮ' if keep else 'KHÔNG GIỮ'} DeiT làm mô hình đơn theo tiêu chí "
      f"đã đặt trước.")
print("   (DeiT yếu hơn vẫn có thể giữ làm thành viên ensemble nếu nó sửa được "
      "những ca khác.)")
if RUN_MODE == "smoke":
    print("   smoke run — chỉ kiểm tra pipeline")

,model,auc,hsas,sensitivity,specificity,fp
0,resnet18 B1,0.9779,NaN,0.9951,0.7689,52
1,densenet121,0.9801,NaN,0.9951,0.8044,44
2,ens R+D,0.9792,NaN,0.9951,0.8222,40
3,deit_small,0.9789,0.8009,0.9951,0.6578,77



TN 148  FP 77  FN 1  TP 202
khoảng cách OOF→benchmark, độ đặc hiệu: +0.3275

=> KHÔNG GIỮ DeiT làm mô hình đơn theo tiêu chí đã đặt trước.
   (DeiT yếu hơn vẫn có thể giữ làm thành viên ensemble nếu nó sửa được những ca khác.)


# 4. Bước tiếp theo

Ensemble cuối cùng, chỉ trung bình xác suất, chọn thành viên và ngưỡng hoàn
toàn bằng OOF:

```
ResNet + DenseNet          (hiện 0,8222)
ResNet + DeiT
DenseNet + DeiT
ResNet + DenseNet + DeiT
```

Không dùng trung bình rank, không đưa mô hình Stage B vào (nó trùng DenseNet),
không tối ưu trọng số trên benchmark.